# CIFAR10 simple Continual DiT CLF

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/CIFAR10%20simple%20Continual%20DiT%20CLF.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Ffiles%2Fnotebooks%2FCIFAR10%2520simple%2520Continual%2520DiT%2520CLF.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Ffiles%2Fnotebooks%2FCIFAR10%20simple%20Continual%20DiT%20CLF.ipynb)
[![Open in Studio Lab](https://img.shields.io/badge/Open_in-Studio_Lab-232F3E)](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/CIFAR10%20simple%20Continual%20DiT%20CLF.ipynb)

Select a GPU runtime in Colab or Kaggle; enable Internet in Kaggle. Binder is suitable for inspecting the notebook. For Studio Lab, set `RUNTIME = "studiolab"` in the first code cell and use a GPU session for training.

These buttons open the published GitHub `main` version. Publish this notebook and its project setup files before using the links, or upload the notebook directly to Colab/Kaggle while it remains local. The first code cell prepares the project runtime; if it requests a kernel restart, restart and run the cells again.

One joint DiT learns two CIFAR-10 classes at a time. It is both the classifier and the replay generator. Training uses classification and denoising losses, with no distillation, teacher model, or semantic consolidation. Evaluation uses only the official test split.

The model settings follow the clean-trained, no-distillation **Copy194 / recipe165**: 85.94% recorded ordinary clean-test accuracy in its original single-task experiment. This was the strongest seed in the retained no-distillation clean confirmation group (five-seed mean 85.588%). The continual experiment below is a new protocol, not a claim of reproducing that score.

In [ ]:
"""Prepare a notebook checkout and runtime before importing scientific packages.

This canonical first-cell source is copied into maintained notebooks and generated
HPO notebooks. Edit REVISION/RUNTIME/CUDA before execution when needed. Existing
checkouts are reused without fetching; only missing setup code is downloaded.
ROOT is the selected absolute pathlib.Path; RUNTIME_PACKAGES maps package names
to installed version strings. Preparation may clone/install under the selected
policy and changes cwd/import paths through files.notebooks.init.prepare_notebook.
"""
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"
CUDA = None

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME, 
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "files" / "notebooks" / "init.py" for path in _locations
                     if (path / "files" / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/files/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, 
    revision=REVISION, 
    runtime=RUNTIME, 
    cuda=CUDA
)

In [ ]:
from common import utils


utils.init()

## Experiment settings

Five tasks use the fixed random order from our previous experiments. The model seed is 505, matching Copy194; the data-loader seed is 42. Epochs and batch size retain the continual learner's defaults. Run the notebook on a remote GPU with TensorFlow 2.20.0 / Keras 3.11.2.

In [ ]:
import tensorflow as tf
from common.runtime import configure_runtime


epochs = 100
batch_size = 128
seed = 505
data_seed = 42
task_size = 2
class_order = [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
task_groups = [[7, 3], [2, 8], [5, 6], [9, 4], [0, 1]]
output_dir = ROOT / "files/results/cifar10_simple_continual_dit_clf_seed505"

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
if not gpus:
    raise RuntimeError("Run the training cells on a remote GPU runtime.")
configure_runtime(dtype_policy="float32", seed=seed)
print("Epochs per task:", epochs, "| Batch size:", batch_size)
print("Task schedule:", task_groups)
print("Model seed:", seed, "| Data seed:", data_seed)

## CIFAR10 data

Use all 50,000 official training images as the available training pool and all 10,000 official test images for evaluation. There is no validation split. The learner selects each task's classes from these arrays and later applies its default training-pool sampling.

Pixels remain in [0, 255] at the data/replay boundary; the wrapper handles normalization. The preview load below prints the actual data shapes. The learner reloads the same cached dataset when training starts.

In [ ]:
from common.dataloader import load_cifar10


dataset_settings = {
    "validation_ratio": 0.0, 
    "preprocess": None, 
    "onehot_labels": False, 
    "seed": data_seed
}
x_train, y_train, _, _, x_test, y_test = load_cifar10(
    indices=None, verbose=1, **dataset_settings
)
print("Training images:", x_train.shape, "| Labels:", y_train.shape)
print("Test images:", x_test.shape, "| Labels:", y_test.shape)
print("Validation split: none")
class_names = [
    "airplane", "automobile", "bird", "cat", "deer", 
    "dog", "frog", "horse", "ship", "truck"
]
for task, group in enumerate(task_groups, start=1):
    print(f"Task {task}: {group} -> {[class_names[c] for c in group]}")

## DiT denoiser and classifier

Copy194 uses eight denoising transformer blocks and eight classifier blocks, width 128, four attention heads, convolutional 2-by-2 patch embedding, classifier DropPath 0.05, and head dropout 0.5. There is no hidden classifier MLP projection.

`num_classes=None` is the continual-learning adaptation: the class vocabulary grows from two to ten instead of exposing all ten classes at initialization. The classifier uses clean images and null class conditioning. The denoiser still learns the full 1,000-timestep diffusion process.

In [ ]:
from diffusion.models.transformer.di_t_classifier import DiTClassifier
from diffusion.models.wrapper.diffusion_classifier import DiffusionClassifier


configure_runtime(dtype_policy="float32", seed=seed)
vit = DiTClassifier(
    image_size=32, channels=3, num_classes=None, 
    patch_size=2, patchify_with_cnn=True, 
    depth=8, dim=128, dim_forced=True, 
    mha_num_heads=4, vit_block_mlp_ratio=4.0, 
    droppath_rate=0.0, vit_block_dropout_rate=0.0, 
    vit_block_attention_dropout_rate=0.0, 
    clf_depth=8, clf_dim=None, clf_dim_forced=False, 
    clf_mha_num_heads=4, clf_vit_block_mlp_ratio=4.0, 
    clf_droppath_rate=0.05, clf_vit_block_dropout_rate=0.0, 
    clf_vit_block_attention_dropout_rate=0.0, 
    classifier_dropout_rate=0.5, classifier_mlp_ratio=None, 
    classifier_mlp_activation_func="tanh", 
    classifier_only_cls_token=True, classifier_only_distil_token=True, 
    cls_token_type=None, distil_token_type=None, 
    clf_cls_token_type="new_weight", clf_distil_token_type=None, 
    feature_aggregation_ids_dict={1: [-1]}, clf_connection_ids_dict={-1: [-1]}, 
    force_global_avg_pooling=True, 
    cond_type="time_label", clf_cond_type="time_label", conds_merger_type="add", 
    patches_pos_embed_type="2d_sincos", patches_pos_merger_type="add", 
    time_embed_type="1d_sincos", time_embed_trainable=False, 
    label_embed_type="new_weight", label_embed_trainable=False, 
    timesteps=1000, use_cfg=True, use_refiner_cnn=False, 
    seed=seed
)
vit.summary()

In [ ]:
model = DiffusionClassifier(
    network=vit, use_ema=False, 
    preprocess_type="standardize", scheduler_name="clipped_cosine", 
    modify_first_t=True, p_uncond=0.1, train_noisified_max_timesteps=1000, 
    noise_loss_coef=1.0, image_loss_coef=0.0, noise_distil_loss_coef=0.0, 
    clf_loss_coef=0.00215, clf_distil_loss_coef=0.0, 
    clf_acc_coef=1.0, clf_distil_acc_coef=0.0, ctr_acc_coef=0.0, 
    clf_train_class_input_type="null_class_only", 
    clf_train_noisy_input_type="noisy", 
    clf_train_noisified_max_timesteps=0, clf_test_noisified_max_timesteps=0, 
    mask_by_nulls=False, mask_by_t_threshold=False, 
    teacher_network=None, classifier_teacher_network=None, noise_teacher_network=None, 
    trainable_teacher=False, 
    test_network_name="raw", test_steps=100, test_cfg_scale=3.0, test_eta=0.0, 
    seed=seed
)
print("Joint objective: noise MSE + 0.00215 * classification CE")
print("Classifier training and testing: clean images (noise caps = 0)")
print("Distillation: off | Teachers: none | Semantic consolidation: off")
print("Replay generator: this same raw DiT | DDIM steps: 100 | CFG: 3 | eta: 0")

## Default continual settings

These values are the existing learner defaults, made explicit so the training exposure is visible. **The default is not to train on the entire combined pool.**

| Setting | Default and effect |
|---|---|
| `remove_prev_classes=True` | Only the current two classes contribute real training images. |
| `keep_same_model=True` | Retain learned weights as new classes arrive. |
| `use_generative_replay=True` | Generate old examples with the joint DiT before the new task is trained. |
| `replay_budget_mode='legacy'` | Generate 1,000 examples per old class: 0 / 2,000 / 4,000 / 6,000 / 8,000 across the five tasks. |
| `train_num=1000` | Draw 1,000 rows **with replacement** from the combined current/replay pool once per task; reuse this sampled pool across epochs. |
| `replay_selection='all'`, multiplier 1 | No quality filtering or extra candidate generation. |
| `use_buffer=False`, cache off | No real-image replay buffer or replay cache. |
| 100 epochs, batch 128 | Eight batches per epoch from the 1,000-row sampled pool; the final partial batch is retained. |
| `callback_patience=None` | This joint-training path has no built-in early stopping. |
| Ensemble evaluation off | Report ordinary clean-image classifier accuracy. |
| Task checkpoints off | Save final weights in the later save cell; automatic task-boundary recovery is not enabled. |

Each task initially offers 10,000 current real images. Later tasks add the generated old pool before the 1,000-row draw, so the realized old/new proportions are sampled, not fixed. No extra image augmentation or padding is added. The requested task order, seed inherited from Copy194, joint-model classifier, test-only evaluation, and verbose reporting are stated separately.

In [ ]:
generative_model_kwargs = {"train_num": 1000, "samples_per_class": 1000}
print("Generator exposure defaults:", generative_model_kwargs)
for task in range(len(task_groups)):
    old_classes = task * task_size
    generated_rows = old_classes * generative_model_kwargs["samples_per_class"]
    print(
        f"Task {task + 1}: 10000 current real + {generated_rows} generated old -> "
        f"{generative_model_kwargs['train_num']} training draws with replacement"
    )

## Optimizer

Retain Copy194's Adam and initial learning rate 0.001 with cosine decay. The optimizer continues across tasks, so its cosine horizon covers the **whole five-task run**: 5 tasks × 100 epochs × 8 batches = 4,000 updates. This is the schedule adaptation for continual training; it does not reset or reach zero after task 1.

In [ ]:
from tensorflow.keras import optimizers


steps_per_epoch = (generative_model_kwargs["train_num"] + batch_size - 1) // batch_size
decay_steps = len(task_groups) * epochs * steps_per_epoch
lr_schedule = optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=decay_steps
)
model.compile(optimizer=optimizers.Adam(lr_schedule), loss="mse")
print("Optimizer: Adam | Initial learning rate: 0.001")
print("Global cosine horizon:", decay_steps, "updates")

In [ ]:
from common.callbacks.lr_logger import LrLogger


callbacks_list = [LrLogger()]
print("Verbose fit progress and learning-rate logging enabled.")
print("Task model summaries, replay progress, previews, and final plots enabled.")

## Continual training

Each task expands the model's vocabulary, generates replay for old classes, samples the default training pool, and jointly trains the classifier and denoiser. There is no teacher fitting or semantic phase. Test evaluation follows every completed task and includes all classes seen so far, without supplying the task identity to the classifier.

`validation_ratio=0.0` and `use_valset=False` mean no validation data is passed to fitting. The official test set is evaluated after each task, not after every epoch. `verbose=1` enables batch/epoch progress and task/replay messages; the internal dataset loader and task-matrix prediction helper do not expose their own progress bars.

In [ ]:
from common.learner import continually_learn


output_dir.mkdir(parents=True, exist_ok=True)
print("Starting five-task continual training. Results directory:", output_dir)
results = continually_learn(
    class_num=10, load_dataset_fn=load_cifar10, 
    load_dataset_fn_kwargs=dataset_settings, 
    class_order=class_order, task_groups=task_groups, task_size=task_size, 
    class_order_mode="fixed", task_order_mode="fixed", 
    generative_model=model, use_generative_model_classifier=True, 
    generative_model_kwargs=generative_model_kwargs, 
    remove_prev_classes=True, keep_same_model=True, 
    use_generative_replay=True, replay_budget_mode="legacy", 
    replay_selection="all", replay_candidate_multiplier=1, replay_cache_mode="off", 
    use_buffer=False, use_distillation=False, train_classifier_separately=False, 
    epochs=epochs, batch_size=batch_size, callback_patience=None, 
    use_valset=False, experiment_phase="legacy", 
    use_ensemble_accuracy=False, evaluate_ensemble_accuracy=False, 
    save_task_checkpoints=False, 
    plot_results=True, show_generated_images=True, show_network_summary=True, 
    callbacks_list=callbacks_list, return_details=True, 
    verbose=1, seed=seed
)
print("Completed task groups:", results["task_classes"])
print("Official test evaluated:", results["test_evaluated"])

## Save weights

Save the final joint DiT weights and its expanded model configuration. Automatic task checkpoints remain disabled, matching the learner default; these final files are not a full interrupted-training recovery checkpoint.

In [ ]:
model = results["generative_model"]
model.save_weights(str(output_dir / "final.weights.h5"))
(output_dir / "model.json").write_text(model.to_json(), encoding="utf-8")
print("Saved final weights and model configuration to:", output_dir)

## Test results

Rows identify the completed training task; columns identify the test task. Every prediction competes against all seen classes. Future-task cells are unavailable. Accuracy is shown as a percentage; forgetting and backward transfer are shown in percentage points. No validation metrics are reported.

In [ ]:
import pandas as pd


task_labels = [f"Task {i}" for i in range(1, len(task_groups) + 1)]
test_matrix = pd.DataFrame(
    results["ordinary_accuracy_matrix"], index=task_labels, columns=task_labels
)
test_matrix.to_csv(output_dir / "test_accuracy_matrix.csv")
print("Official test accuracy matrix (%)")
print((100 * test_matrix).round(2).to_string())

metrics = pd.Series(results["continual_metrics"], name="Official test")
metrics.to_csv(output_dir / "continual_metrics.csv")
print("Continual accuracy (%), forgetting and backward transfer (percentage points)")
print((100 * metrics).round(2).to_string())

trajectory = pd.DataFrame({
    "Old-task accuracy": results["old_task_accuracy"], 
    "New-task accuracy": results["new_task_accuracy"]
}, index=task_labels)
trajectory.to_csv(output_dir / "old_new_test_accuracy.csv")
print("Old/new task test accuracy (%)")
print((100 * trajectory).round(2).to_string())

## Class-conditional samples

Show one generated image per learned class from the same raw joint DiT. The sampler's default labels select real classes and skip the CFG null condition. Titles follow the fixed learning order.

In [ ]:
print("Sampling one image per class: DDIM100, CFG3, eta0")
samples = model.sample(
    network_name="raw", labels=None, steps=100, scale=3.0, eta=0.0, 
    verbose=True, seed=data_seed
)
utils.plot_images(
    samples, row=2, col=5, 
    save_path=output_dir / "final_samples.png", 
    titles=[class_names[c] for c in class_order], value_range=(0, 255)
)
print("Saved sample grid:", output_dir / "final_samples.png")

## Training history

Save and plot each task's joint-training history. These curves contain training metrics; test performance is recorded separately at the task boundaries above.

In [ ]:
for task, history in enumerate(results["generative_histories"], start=1):
    history_path = output_dir / f"task_{task}_history.csv"
    pd.DataFrame(history).to_csv(history_path, index_label="epoch_index")
    print(f"Task {task} training history: {history_path}")
    utils.plot_history(history, show_all_x_ticks=False)

## Reference and adaptations

Model reference: [executed Copy194](DiT%20HPO%20Codex/CIFAR10%20DiT%20CLF-Copy194.ipynb), [resolved configuration](DiT%20HPO%20Codex/exports/final20_completed_20261010/results/copy194/config.json), and [per-seed results](DiT%20HPO%20Codex/exports/final20_completed_20261010/per_seed.csv). Higher historical scores involving distillation or different protocols are not the basis of this no-distillation recipe.

The reference used a fixed ten-class model, a training/development split, and single-task training. Here the vocabulary grows, the validation split is removed, default continual replay/resampling is used, and the cosine schedule spans the whole task sequence. The historical score is not a measured result of this notebook. No executed outputs or trained weights are included.